In [ ]:
import os
import pandas as pd

# =========================
# 1. 경로 설정
# =========================
BASE_DIR = r"D:\캡스톤"

base_path = os.path.join(BASE_DIR, "10개년_강남구_gis.xlsx")
info_path = os.path.join(BASE_DIR, "0630_베이스.xlsx")

save_path = os.path.join(BASE_DIR, "10개년_강남구_gis_단지정보추가.xlsx")


# =========================
# 2. 데이터 불러오기
# =========================
base_df = pd.read_excel(base_path)
info_df = pd.read_excel(info_path)

print("베이스 데이터 크기:", base_df.shape)
print("추가정보 데이터 크기:", info_df.shape)

print("\n베이스 컬럼:")
print(base_df.columns.tolist())

print("\n추가정보 컬럼:")
print(info_df.columns.tolist())


# =========================
# 3. 매칭 기준 및 가져올 컬럼 설정
# =========================
merge_keys = ["시군구", "행정동", "단지명"]

add_cols = [
    "경과연수",
    "세대수",
    "세대당_주차대수",
    "용적률",
    "건폐율",
    "거래층",
    "아파트_브랜드",
    "대지면적(㎡)",
    "건축면적(㎡)",
    "연면적(㎡)",
    "용적률산정연면적(㎡)",
    "총주차수"
]


# =========================
# 4. 컬럼 존재 여부 확인
# =========================
required_cols = merge_keys + add_cols

missing_in_base = [col for col in merge_keys if col not in base_df.columns]
missing_in_info = [col for col in required_cols if col not in info_df.columns]

if missing_in_base:
    raise ValueError(f"베이스 데이터에 없는 매칭 컬럼: {missing_in_base}")

if missing_in_info:
    raise ValueError(f"0630_베이스 데이터에 없는 컬럼: {missing_in_info}")


# =========================
# 5. 매칭 key 정리
# =========================
for col in merge_keys:
    base_df[col] = base_df[col].astype(str).str.strip()
    info_df[col] = info_df[col].astype(str).str.strip()


# =========================
# 6. 0630_베이스에서 필요한 컬럼만 추출
# =========================
info_merge = info_df[merge_keys + add_cols].copy()


# =========================
# 7. 0630_베이스 내 중복 key 처리
# =========================
dup_count = info_merge.duplicated(subset=merge_keys).sum()
print("\n0630_베이스 내 중복 key 개수:", dup_count)

if dup_count > 0:
    print("중복 key가 있어 첫 번째 행만 사용합니다.")
    info_merge = info_merge.drop_duplicates(subset=merge_keys, keep="first")


# =========================
# 8. 베이스 데이터 기준 병합
# =========================
merged_df = base_df.merge(
    info_merge,
    on=merge_keys,
    how="left"
)

print("\n병합 후 데이터 크기:", merged_df.shape)


# =========================
# 9. 매칭 실패 확인
# =========================
check_col = add_cols[0]  # 경과연수 기준으로 확인
missing_count = merged_df[check_col].isna().sum()

print(f"\n매칭 실패 행 개수({check_col} NA 기준):", missing_count)
print("매칭 실패 비율:", round(missing_count / len(merged_df) * 100, 2), "%")

print("\n매칭 실패 예시:")
print(
    merged_df[merged_df[check_col].isna()][merge_keys]
    .drop_duplicates()
    .head(20)
)


# =========================
# 10. 저장
# =========================
merged_df.to_excel(save_path, index=False)

print("\n저장 완료!")
print("저장 경로:", save_path)
print("최종 데이터 크기:", merged_df.shape)

베이스 데이터 크기: (39350, 28)
추가정보 데이터 크기: (15230, 31)

베이스 컬럼:
['시군구', '경도', '위도', '행정동', '단지명', '전용면적(㎡)', '계약년월', '계약년', '계약월', '계약일', '거래금액(만원)', '층', '건축년도', '도로명', 'subway_nearest_dist', 'subway_500m_count', 'bus_stop_nearest_dist', 'bus_stop_500m_count', 'school_nearest_dist', 'school_750m_count', 'park_nearest_dist', 'park_750m_count', 'culture_nearest_dist', 'culture_1000m_count', 'commercial_nearest_dist', 'commercial_750m_count', 'hospital_nearest_dist', 'hospital_750m_count']

추가정보 컬럼:
['시군구', '매핑키', '자치구', '행정동', '행정동코드', '번지', '본번', '부번', '단지명', '면적', '단지내_동수', '거래시점', '분기', '경과연수', '세대수', '세대당_주차대수', '용적률', '건폐율', '거래층', '경도', '아파트_브랜드', '대지면적(㎡)', '건축면적(㎡)', '연면적(㎡)', '용적률산정연면적(㎡)', '총주차수', '사용승인일', '건물명', '주용도코드명', '기타용도', '공동주택_매칭여부']


ValueError: 0630_베이스 데이터에 없는 컬럼: ['대지면적', '건축면적']